In [1]:
import pandas as pd
import os

In [4]:
BRUTES = r"Brute\fr_EOHL_Reddit_clean.xlsx"
EMOTION = r"corpus_emotyc_reddit_features.csv"

In [5]:
df_brute = pd.read_excel(BRUTES)
df_emotion = pd.read_csv(EMOTION,index_col=0)

In [6]:
df_brute.drop(columns=['id','date'], inplace=True)

In [7]:
df_brute.head(5)

,user,text,title,Perspective_label
0,joaobatista93,"C'est dingue, après tout ce temps, plus de 15 ...",NaN,0
1,Tavek,"Plutôt d'accord pour les SAV, c'est le but mêm...",NaN,0
2,AskingToFeminists,Bordel. J'avais tapé une longue réponse. Une f...,NaN,0
3,Various-Adagio6411,Non pas de problème de ce côté là heureusement !,NaN,0
4,NewPhoneSuisse,Les violences étaient clairement motivées par ...,NaN,0


In [8]:
df_brute.columns

Index(['user', 'text', 'title', 'Perspective_label'], dtype='object')

In [9]:
df_brute["text"] = df_brute["text"].astype(str)

In [14]:
df_emotion.columns

Index(['feature_found', 'avec_emo_comportementale', 'avec_cat_fierte',
       'avec_emo_complexe', 'avec_cat_colere', 'avec_cat_joie',
       'avec_cat_autre', 'avec_cat_tristesse', 'avec_emo', 'avec_cat_surprise',
       'avec_emo_designee', 'avec_emo_base', 'avec_cat_peur',
       'avec_cat_admiration', 'avec_emo_montree', 'avec_emo_suggeree',
       'avec_cat_embarras', 'text'],
      dtype='object')

In [16]:
last_line = df_emotion.iloc[-1,17]

In [17]:
print(last_line)

Uleul y'ooooo-ooonson   Lïlealol élèleliel  Talalalaaaaa laaaaa laaaaaaaa...


In [18]:
import re
import unicodedata

def normalize_whitespace(text: str) -> str:
    text = str(text)
    text = unicodedata.normalize('NFKC', text)   # apostrophes courbes, guillemets, etc.
    text = re.sub(r'[\s\xa0\u200b]+', ' ', text) # espaces, sauts de ligne, insécables
    return text.strip()


In [19]:
print(normalize_whitespace(last_line))

Uleul y'ooooo-ooonson Lïlealol élèleliel Talalalaaaaa laaaaa laaaaaaaa...


In [20]:
df_brute.columns

Index(['user', 'text', 'title', 'Perspective_label'], dtype='object')

In [21]:
for idx, text in enumerate(df_brute.iloc[:,1]):
    if normalize_whitespace(last_line) in normalize_whitespace(text):
        print(idx)
        break

998


In [22]:
print(df_brute.iloc[998,1])

Putain tu me l'as mis en tête, salopard.

Uleul y'ooooo-ooonson  
Lïlealol élèleliel

Talalalaaaaa laaaaa laaaaaaaa...


In [23]:
df_brute[df_brute["title"].notna()]

,user,text,title,Perspective_label
29,latestasianews,nan,Wall Street's Top Analysts Are Bullish on Thes...,0
32,HermaeusMorah,nan,Métropole de Lyon : les Verts veulent créer de...,0
34,latestasianews,nan,Images of Queen Elizabeth II: A life in pictures,0
88,xroche,nan,"En Israël, les déclarations de Sergueï Lavrov ...",0
89,[deleted],nan,L’étonnante bande-son d’un séisme record enreg...,0
...,...,...,...,...
19864,demauroy,nan,Perturber un événement sportif pourrait être d...,0
19882,maxence1994,nan,Cinq des sept raffineries de France métropolit...,0
19921,The_tricksterz,nan,Un conseil d'utilité publique,0
19944,Hedowas,"Spoiler alert: je suis un nouveau-né naïf, mai...","[RANT] Passer le permis rapidement, une désill...",0


In [24]:
def make_full_text(row):
    title = normalize_whitespace(row["title"]) if pd.notna(row["title"]) else ""
    text  = normalize_whitespace(row["text"])
    return (title + " " + text).strip() if title else text

df_brute["full_text"] = df_brute.apply(make_full_text, axis=1)


In [25]:
from tqdm import tqdm

brute_full = [
    normalize_whitespace(
        (str(row["title"]) if pd.notna(row["title"]) else "") + " " + str(row["text"])
    )
    for _, row in df_brute.iterrows()
]
brute_ids = list(df_brute.index)

emote = dict()
current_pos = 0

for idx, line in tqdm(enumerate(df_emotion.iloc[:, 17]), total=len(df_emotion)):
    needle = normalize_whitespace(str(line))
    found = False
    for j in range(current_pos, len(brute_full)):
        if needle in brute_full[j]:
            emote[idx] = brute_ids[j]
            current_pos = j  # on reste à j, pas j+1
            found = True
            break
    if not found:
        emote[idx] = None

not_found = [idx for idx, v in emote.items() if v is None]
print(f"Trouvés : {len(emote) - len(not_found)} / {len(emote)}")
print(f"Non trouvés : {len(not_found)}")


100%|██████████| 2896/2896 [00:00<00:00, 20652.85it/s]

Trouvés : 2888 / 2896
Non trouvés : 8


In [26]:
df_emotion.columns

Index(['feature_found', 'avec_emo_comportementale', 'avec_cat_fierte',
       'avec_emo_complexe', 'avec_cat_colere', 'avec_cat_joie',
       'avec_cat_autre', 'avec_cat_tristesse', 'avec_emo', 'avec_cat_surprise',
       'avec_emo_designee', 'avec_emo_base', 'avec_cat_peur',
       'avec_cat_admiration', 'avec_emo_montree', 'avec_emo_suggeree',
       'avec_cat_embarras', 'text'],
      dtype='object')

In [27]:
print(normalize_whitespace(df_emotion.iloc[10,17]))

Edit: recall the violence and sexual harassment I was subjected to growing up for being "unmanly"


In [28]:
print(normalize_whitespace(df_brute.iloc[4,1]))

Les violences étaient clairement motivées par l'homophobie et la misogynie, pourtant je n'ai admis que j'étais trans (et lesbienne) qu'à 29 ans. Pourquoi "clairement" ? A l'époque, si je comprends bien, tu t'identifiais en tant que garçon, et tu étais intéressé par les filles. Edit: I recall the violence and sexual harassment I was subjected to growing up for being "unmanly" Je vois donc comment tu peux penser avoir reçu un harcèlement misogyne, mais en quoi ce harcèlement était homophobe ?


In [29]:
print(normalize_whitespace(df_emotion.iloc[10,17]) in normalize_whitespace(df_brute.iloc[4,1]))

False


In [31]:
for key ,val in emote.items():
    if val is None:
        print(f"{key} : {df_emotion.iloc[key,17]}")

10 : Edit:  recall the violence and sexual harassment I was subjected to growing up for being "unmanly"
188 : On nous reproche souvent : « Mais pourquoi, à Charlie, êtes-vous si critiques à l’égard de La France insoumise et de son leader?
194 : On peut nous demander beaucoup de choses : d’être plus drôles, d’être moins cons, d’être plus intelligents, d’être plus radicaux, ou que sais-je encore?
213 : Eh bien, pas du tout, car pour gagner les 600000 voix qui avaient manqué au leader de La France insoumise et qui lui auraient permis d’arriver au second tour de la présidentielle de 2017, ce parti décida d’aller les chercher dans les quartiers où la population musulmane est importante, ce qui a très bien fonctionné au vu des résultats obtenus dans certaines circonscriptions de la banlieue parisienne.
214 : Face à la perspective de réaliser de bons scores dans ces départements, que vaut la question du communautarisme et du repli identitaire?
227 : Qui aurait la folie de vouloir s’opposer à 

In [32]:
not_found

[10, 188, 194, 213, 214, 227, 1710, 2393]

In [33]:
df_emotion.iloc[not_found]

,feature_found,avec_emo_comportementale,avec_cat_fierte,avec_emo_complexe,avec_cat_colere,avec_cat_joie,avec_cat_autre,avec_cat_tristesse,avec_emo,avec_cat_surprise,avec_emo_designee,avec_emo_base,avec_cat_peur,avec_cat_admiration,avec_emo_montree,avec_emo_suggeree,avec_cat_embarras,text
file_id,,,,,,,,,,,,,,,,,,
11,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Edit: recall the violence and sexual harassme...
189,True,1,0,0,1,0,0,0,1,0,0,1,0,0,0,1,0,"On nous reproche souvent : « Mais pourquoi, à ..."
195,True,0,0,0,0,1,0,0,1,0,1,1,0,0,0,0,0,On peut nous demander beaucoup de choses : d’ê...
214,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"Eh bien, pas du tout, car pour gagner les 6000..."
215,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Face à la perspective de réaliser de bons scor...
228,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Qui aurait la folie de vouloir s’opposer à cela?
1711,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,* [Les fils précédents] * [Les megathreads] ...
2394,True,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,"C'est simple, c'est enfantin, mais ca me ferai..."


In [34]:
for t in df_emotion.iloc[not_found,17]:
    print(f"{t}\n")

Edit:  recall the violence and sexual harassment I was subjected to growing up for being "unmanly"

On nous reproche souvent : « Mais pourquoi, à Charlie, êtes-vous si critiques à l’égard de La France insoumise et de son leader?

On peut nous demander beaucoup de choses : d’être plus drôles, d’être moins cons, d’être plus intelligents, d’être plus radicaux, ou que sais-je encore?

Eh bien, pas du tout, car pour gagner les 600000 voix qui avaient manqué au leader de La France insoumise et qui lui auraient permis d’arriver au second tour de la présidentielle de 2017, ce parti décida d’aller les chercher dans les quartiers où la population musulmane est importante, ce qui a très bien fonctionné au vu des résultats obtenus dans certaines circonscriptions de la banlieue parisienne.

Face à la perspective de réaliser de bons scores dans ces départements, que vaut la question du communautarisme et du repli identitaire?

Qui aurait la folie de vouloir s’opposer à cela?

* [Les fils précédents]

In [37]:

# Pour chaque texte d'émotion non trouvé : liste de tuples (un tuple par mot)
# Puis on cherche le brute row dont le full_text contient le plus de ces mots

def text_to_word_tuples(text: str) -> list[tuple]:
    words = normalize_whitespace(text).lower().split()
    return [(w,) for w in words if w]

# Pré-calcul : full_text de chaque brute en set de mots (minuscules)
brute_word_sets = [
    set(normalize_whitespace(ft).lower().split())
    for ft in df_brute["full_text"]
]

best_ids = []  # liste des brute IDs les plus proches pour chaque not_found

for idx in not_found:
    emotion_text = df_emotion.iloc[idx, 17]
    word_tuples = text_to_word_tuples(str(emotion_text))
    emotion_words = {t[0] for t in word_tuples}  # set de mots

    best_id, best_count = None, 0
    for j, bws in enumerate(brute_word_sets):
        count = len(emotion_words & bws)
        if count > best_count:
            best_count = count
            best_id = brute_ids[j]

    best_ids.append(best_id)
    print(f"emotion[{idx}] → brute_id={best_id}  ({best_count} mots communs)")

print("\nbest_ids:", best_ids)


emotion[10] → brute_id=4  (16 mots communs)
emotion[188] → brute_id=65  (20 mots communs)
emotion[194] → brute_id=65  (18 mots communs)
emotion[213] → brute_id=65  (52 mots communs)
emotion[214] → brute_id=65  (18 mots communs)
emotion[227] → brute_id=65  (8 mots communs)
emotion[1710] → brute_id=588  (20 mots communs)
emotion[2393] → brute_id=838  (23 mots communs)

best_ids: [4, 65, 65, 65, 65, 65, 588, 838]


In [38]:
for idx,id in enumerate(best_ids):
    emote[not_found[idx]] = id


In [39]:
# Vérification : la liste mergée des brute_ids (dans l'ordre des indices émotion) est-elle triée ?
merged = [emote[i] for i in sorted(emote) if emote[i] is not None]

is_sorted = all(merged[i] <= merged[i+1] for i in range(len(merged)-1))

if is_sorted:
    print(f"OK — la liste mergée de {len(merged)} brute_ids est triée en ordre croissant.")
else:
    violations = [(i, merged[i], merged[i+1]) for i in range(len(merged)-1) if merged[i] > merged[i+1]]
    print(f"NON trié — {len(violations)} violation(s) :")
    for i, a, b in violations[:10]:
        print(f"  position {i} : brute_id {a} > {b}")


OK — la liste mergée de 2896 brute_ids est triée en ordre croissant.


In [40]:
df_emotion.columns

Index(['feature_found', 'avec_emo_comportementale', 'avec_cat_fierte',
       'avec_emo_complexe', 'avec_cat_colere', 'avec_cat_joie',
       'avec_cat_autre', 'avec_cat_tristesse', 'avec_emo', 'avec_cat_surprise',
       'avec_emo_designee', 'avec_emo_base', 'avec_cat_peur',
       'avec_cat_admiration', 'avec_emo_montree', 'avec_emo_suggeree',
       'avec_cat_embarras', 'text'],
      dtype='object')

In [41]:
df_brute_without_emotion = df_brute[999:]
df_brute = df_brute.iloc[:999]

In [42]:
from collections import defaultdict

EMO_COLS = [
    'avec_emo_comportementale', 'avec_cat_fierte', 'avec_emo_complexe',
    'avec_cat_colere', 'avec_cat_joie', 'avec_cat_autre', 'avec_cat_tristesse',
    'avec_emo', 'avec_cat_surprise', 'avec_emo_designee', 'avec_emo_base',
    'avec_cat_peur', 'avec_cat_admiration', 'avec_emo_montree',
    'avec_emo_suggeree', 'avec_cat_embarras'
]

# inverser emote : emo_idx -> brute_id  =>  brute_id -> [emo_idx, ...]
brute_to_emo = defaultdict(list)
for emo_idx, brute_id in emote.items():
    if brute_id is not None:
        brute_to_emo[brute_id].append(emo_idx)

for col in EMO_COLS:
    df_brute[col] = [
        int(df_emotion.iloc[brute_to_emo[brute_id]][col].any())
        if brute_to_emo.get(brute_id)
        else 0
        for brute_id in df_brute.index
    ]

df_brute[EMO_COLS].sum()


avec_emo_comportementale     59
avec_cat_fierte               8
avec_emo_complexe            38
avec_cat_colere             148
avec_cat_joie                87
avec_cat_autre                7
avec_cat_tristesse           75
avec_emo                    422
avec_cat_surprise            55
avec_emo_designee           140
avec_emo_base               321
avec_cat_peur                59
avec_cat_admiration         156
avec_emo_montree            118
avec_emo_suggeree           151
avec_cat_embarras            12
dtype: int64

In [45]:
df_brute.head(1)

,user,text,title,Perspective_label,full_text,avec_emo_comportementale,avec_cat_fierte,avec_emo_complexe,avec_cat_colere,avec_cat_joie,...,avec_cat_tristesse,avec_emo,avec_cat_surprise,avec_emo_designee,avec_emo_base,avec_cat_peur,avec_cat_admiration,avec_emo_montree,avec_emo_suggeree,avec_cat_embarras
0,joaobatista93,"C'est dingue, après tout ce temps, plus de 15 ...",NaN,0,"C'est dingue, après tout ce temps, plus de 15 ...",1,0,0,0,1,...,0,1,0,0,1,0,0,0,0,0


In [43]:
df_brute["Perspective_label"].value_counts()

Perspective_label
0    987
1     12
Name: count, dtype: int64

In [46]:
df_brute_without_emotion["Perspective_label"].value_counts()

Perspective_label
0      18728
1        272
284        1
Name: count, dtype: int64

In [47]:
df_brute_without_emotion[df_brute_without_emotion["Perspective_label"]==284]

,user,text,title,Perspective_label,full_text
19999,Kemro59,"Le gaz a l'époque c'était du gaz de ville, dif...",NaN,284,"Le gaz a l'époque c'était du gaz de ville, dif..."


In [48]:
df_brute_without_emotion  = df_brute_without_emotion[df_brute_without_emotion["Perspective_label"]==1]

In [49]:
for col in EMO_COLS:
    df_brute_without_emotion[col] = 0



In [50]:
df_brute_without_emotion.columns

Index(['user', 'text', 'title', 'Perspective_label', 'full_text',
       'avec_emo_comportementale', 'avec_cat_fierte', 'avec_emo_complexe',
       'avec_cat_colere', 'avec_cat_joie', 'avec_cat_autre',
       'avec_cat_tristesse', 'avec_emo', 'avec_cat_surprise',
       'avec_emo_designee', 'avec_emo_base', 'avec_cat_peur',
       'avec_cat_admiration', 'avec_emo_montree', 'avec_emo_suggeree',
       'avec_cat_embarras'],
      dtype='object')

In [51]:
df_brute.columns

Index(['user', 'text', 'title', 'Perspective_label', 'full_text',
       'avec_emo_comportementale', 'avec_cat_fierte', 'avec_emo_complexe',
       'avec_cat_colere', 'avec_cat_joie', 'avec_cat_autre',
       'avec_cat_tristesse', 'avec_emo', 'avec_cat_surprise',
       'avec_emo_designee', 'avec_emo_base', 'avec_cat_peur',
       'avec_cat_admiration', 'avec_emo_montree', 'avec_emo_suggeree',
       'avec_cat_embarras'],
      dtype='object')

In [52]:
df_brute = pd.concat([df_brute, df_brute_without_emotion], ignore_index=True)

In [53]:
df_brute["found"] = df_brute[EMO_COLS].any(axis=1)
df_brute["found"].value_counts()


found
False    839
True     432
Name: count, dtype: int64

In [54]:
df_brute.columns

Index(['user', 'text', 'title', 'Perspective_label', 'full_text',
       'avec_emo_comportementale', 'avec_cat_fierte', 'avec_emo_complexe',
       'avec_cat_colere', 'avec_cat_joie', 'avec_cat_autre',
       'avec_cat_tristesse', 'avec_emo', 'avec_cat_surprise',
       'avec_emo_designee', 'avec_emo_base', 'avec_cat_peur',
       'avec_cat_admiration', 'avec_emo_montree', 'avec_emo_suggeree',
       'avec_cat_embarras', 'found'],
      dtype='object')

In [55]:
df_brute.to_csv("brute_merged_emotyc.csv", index=False)

In [56]:
df_brute["Perspective_label"].value_counts()

Perspective_label
0    987
1    284
Name: count, dtype: int64